In [ ]:
import pyairbnb
import pandas as pd
import json
import datetime
import os
import time
import random
from typing import List, Dict, Any, Optional

# --- 1. Scraping Function (Unchanged) ---
def scrape_paris_airbnb_listings():
    """Scrapes Airbnb listings for the Paris area and returns the results."""
    
    # Paris bounding box coordinates
    paris_coords = {
        "ne_lat": 48.902,    
        "ne_long": 2.469,    
        "sw_lat": 48.815,    
        "sw_long": 2.224     
    }
    
    os.makedirs('results', exist_ok=True)
    
    try:
        dynamic_hash = pyairbnb.fetch_stays_search_hash()
        results = pyairbnb.search_all(
            check_in="2025-12-01",
            check_out="2025-12-31",
            ne_lat=paris_coords["ne_lat"],
            ne_long=paris_coords["ne_long"],
            sw_lat=paris_coords["sw_lat"],
            sw_long=paris_coords["sw_long"],
            zoom_value=2,
            price_min=0,
            price_max=10000,
            currency="EUR",
            language="fr",
            hash=dynamic_hash
        )        
        
        print(f"Scraper: Found {len(results)} listings.")
        
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        output_filename_json = f"results/search_results_{timestamp}.json"
        
        with open(output_filename_json, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
            
        print(f"Scraper: Saved raw search results to {output_filename_json}")
        return results

    except Exception as e:
        print(f"Scraper Error: An error occurred during search: {e}")
        return []

# --- 2. Amenities Fetching Function (configurable limit) ---
def fetch_amenities_data(room_ids: List[int], detail_limit: Optional[int] = None) -> pd.DataFrame:
    """Fetches detailed amenities for a list of room IDs.

    By default (detail_limit=None) the function will attempt to fetch details for all IDs.
    Set detail_limit to an integer (e.g., 5) for testing to limit the number of detail requests
    and avoid rate limiting during development.
    """

    amenity_data = []
    MAX_RETRIES = 3
    BASE_DELAY = 1  # base delay in seconds between retries

    # Determine which room IDs we will fetch details for
    if detail_limit is None:
        ids_to_fetch = set(room_ids)
        print(f"\nDetails Fetcher: detail_limit=None -> attempting to fetch details for ALL {len(room_ids)} listings.")
    else:
        ids_to_fetch = set(room_ids[:detail_limit])
        print(f"\nDetails Fetcher: detail_limit={detail_limit} -> fetching details for the first {len(ids_to_fetch)} listings (skipping details for the remaining {max(0, len(room_ids) - len(ids_to_fetch))}).")

    for room_id in room_ids:
        # If this ID is not in the to-fetch set, append empty data quickly
        if room_id not in ids_to_fetch:
            amenity_data.append({'id': room_id, 'amenities_list': [], 'amenity_count': 0})
            continue

        # Attempt to fetch details with retries and exponential backoff + jitter
        success = False
        for attempt in range(MAX_RETRIES):
            try:
                details = pyairbnb.get_details(room_id=room_id)
                raw_amenities = details.get('amenities', [])

                # Flatten amenities
                flat_amenities = []
                for category in raw_amenities:
                    for amenity in category.get('values', []):
                        if amenity.get('available'):
                            title = amenity.get('title')
                            if title:
                                flat_amenities.append(title)

                amenity_data.append({
                    'id': room_id,
                    'amenities_list': flat_amenities,
                    'amenity_count': len(flat_amenities)
                })

                print(f"Details Fetcher: Successfully processed ID {room_id} (amenities={len(flat_amenities)}).")

                # polite delay between successful requests (small random jitter)
                time.sleep(BASE_DELAY + random.uniform(0, 0.5))
                success = True
                break

            except Exception as e:
                # exponential backoff with jitter
                delay = BASE_DELAY * (2 ** attempt) + random.uniform(0, 0.5)
                print(f"Error fetching ID {room_id}. Attempt {attempt + 1}/{MAX_RETRIES}. Retrying in {delay:.1f}s. Error: {e}")
                time.sleep(delay)

        if not success:
            print(f"CRITICAL: Failed to fetch details for ID {room_id} after {MAX_RETRIES} attempts. Marking as failed.")
            amenity_data.append({'id': room_id, 'amenities_list': ["API_FETCH_FAILED"], 'amenity_count': 0})

    return pd.DataFrame(amenity_data).set_index('id')

# --- 3. Data Cleaning and Processing (Unchanged) ---
def process_data(listings: List[Dict], amenities_df: pd.DataFrame) -> pd.DataFrame:
    """Normalizes search data, cleans it, and merges the detailed amenities."""

    if not listings:
        print("Processor: No listings to process. Returning empty DataFrame.")
        return pd.DataFrame()

    # 1. Process Search Results
    df = pd.json_normalize(listings)
    
    # 2. Define and rename core columns
    cols_to_keep = {
        "room_id": "id",
        "name": "name",
        "title": "title",
        "price.unit.amount": "price_night",
        "rating.value": "rating",
        "rating.reviewCount": "reviews",
        "coordinates.latitude": "lat",
        "coordinates.longitud": "lon"
    }

    available_cols = {k: v for k, v in cols_to_keep.items() if k in df.columns}
    df = df[list(available_cols.keys())].rename(columns=available_cols)
    
    # Set 'id' as index for merging
    df['id'] = df['id'].astype(int)
    df = df.set_index('id')

    # 3. Merge Amenities
    if not amenities_df.empty:
        df = df.merge(amenities_df, left_index=True, right_index=True, how='left')
        # Fill missing amenity data (for the listings we skipped) with empty lists/zeros
        df['amenities_list'] = df['amenities_list'].apply(lambda x: x if isinstance(x, list) else [])
        df['amenity_count'] = df['amenity_count'].fillna(0).astype(int)
        print(f"Processor: Successfully merged amenities for {len(amenities_df)} listings.")
    else:
        df['amenities_list'] = [[]] * len(df)
        df['amenity_count'] = 0
        print("Processor: Amenities DataFrame was empty; adding empty amenity columns.")

    # 4. Final Cleaning and Type Conversion
    for col in ["price_night", "rating", "reviews"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    
    required_subset = [c for c in ["price_night", "lat", "lon"] if c in df.columns]
    if required_subset:
        original_rows = len(df)
        df = df.dropna(subset=required_subset)
        print(f"Processor: Dropped {original_rows - len(df)} rows with missing essential data.")

    # Reset index for final CSV export
    return df.reset_index()

# --- 4. Main Execution Block ---
if __name__ == "__main__":
    
    print("--- Starting Airbnb Data Pipeline (Amenities Mode) ---")
    
    # 1. Scrape the high-level search data
    raw_listings = scrape_paris_airbnb_listings()
    
    if raw_listings:
        room_ids = [int(listing['room_id']) for listing in raw_listings if 'room_id' in listing]

        # 2. Fetch the detailed amenities
        #    By default we fetch all details (detail_limit=None). For quick testing set detail_limit=5.
        amenities_data = fetch_amenities_data(room_ids, detail_limit=None)

        # 3. Process and merge the data
        cleaned_df = process_data(raw_listings, amenities_data)
        
        # 4. Save the final CSV
        if not cleaned_df.empty:
            output_filename_csv = "paris_airbnb_dataset_TEST_AMENITIES.csv"
            cleaned_df.to_csv(output_filename_csv, index=False)
            print(f"\nFinal: Cleaned Paris Airbnb dataset saved to '{output_filename_csv}'.")
            print("\nCleaned Data Preview (Amenities should now be populated for all fetched IDs):")
            print(cleaned_df[['id', 'name', 'price_night', 'amenity_count', 'amenities_list']].head(10))
        else:
            print("\nFinal: No data saved as the DataFrame is empty.")
    else:
        print("Pipeline halted: Initial scraping yielded no results.")

    print("--- Pipeline Finished ---")

--- Starting Airbnb Data Pipeline (Amenities Test Mode) ---
Scraper: Found 280 listings.
Scraper: Saved raw search results to results/search_results_20251105_002610.json

Details Fetcher: Starting detail fetching for the FIRST 5 listings.
Note: Skipping details for the remaining 275 listings.
Details Fetcher: Successfully processed ID 13492792.
Details Fetcher: Successfully processed ID 923951273538815332.
Details Fetcher: Successfully processed ID 1309224596320319012.
Details Fetcher: Successfully processed ID 971108359050663839.
Details Fetcher: Successfully processed ID 1445613258113090065.
Processor: Successfully merged amenities for 280 listings.
Processor: Dropped 0 rows with missing essential data.

Final: Cleaned Paris Airbnb dataset saved to 'paris_airbnb_dataset_TEST_AMENITIES.csv'.

Cleaned Data Preview (Amenities should only be filled for the first 5 IDs):
                    id                                             name  \
0             13492792    Appartement cosy a